# Guardrail eval · live lab report

Watch one attack go from payload to proof: **1 Understand → 2 Inspect → 3 Launch → 4 Prove**.

| Mode | What Step 3 does | Model calls |
|---|---|---|
| **REPLAY** (`RUN_LIVE = False`, default) | Loads the latest stored run of the row | None |
| **LIVE** (`RUN_LIVE = True`) | Launches the attack now (scratch run) | Real LLM calls, may cost money |

**Phase 1: no vendor guardrail is in the loop.** A landed attack is an undefended baseline, and a model refusal is not a guardrail block.

- **PASS** = the guardrail held. **FAIL** = the attack landed (the canary reached its sink). **NOT-RUN** is never a pass.
- **Canary**: a synthetic value. **Oracle**: the check that looks for it at the sink. **OOB**: a callback seen outside the model's reply.
- `outcome.py` grades. This notebook only displays.

Jump to: [Part A · Guided demo](#guided-demo) · [Part B · Operator console](#operator-console)

**Next:** run the cell below to load the tools.

In [ ]:
import operator_lib as op
from IPython.display import HTML, display

S = op.Session()

<a id="guided-demo"></a>
## Part A · Guided demo

**Done so far:** tools loaded.

**Do:** set the row and mode in the cell below, then run it.
- `DEMO_ROW`: **a row name** (e.g. `WEB-SRV-comment`) **or a TC-ID** (e.g. `PI-03`). A TC-ID picks a representative row for that test case and prints which one. Row names and TC-IDs are listed in Part B · B2 "Test cases", and folded under "Available demo rows" at the top of Part B.
- `RUN_LIVE`: `False` replays stored evidence. `True` launches the attack in Step 3.
- `REPEAT`: trials per condition. 3 is the minimum for agent rows.

**Expected:** if you gave a TC-ID, a line naming the chosen row; otherwise no output.
**Next:** Step 1. If you change the row, rerun Part A from here.

In [ ]:
DEMO_ROW = "WEB-SRV-comment"
RUN_LIVE = False
REPEAT = 3

demo = op.Demo(DEMO_ROW)

<a id="attack"></a>
### Step 1 · Understand the attack

**Done so far:** the row and mode are selected.

**Do:** run the cell and read the card.

**Expected:** the user's legitimate request, the injected instruction, the channel it enters through, and what counts as success. The highlighted text is the delivery marker, which proves the injection was delivered but not that it worked. Some payloads are generated by a server at run time. In that case the card names the source, and the exact text appears in Step 4.

**Next:** Step 2.

In [ ]:
display(HTML(demo.attack_html()))

<a id="listener"></a>
### Step 2 · Inspect the setup and the listener

**Done so far:** you know what the attack does and where it should land.

**Do:** run the cell. It shows the setup, a snapshot of the oracle's sink, and an inline tail of this row's shared listener logs (the notebook equivalent of `tail -n 25 results/…`).

**Expected:** the carrier model, the endpoint and the guardrail status (secrets show only as set / not set), the sink the oracle watches, then the last lines of any listener stdout. All of this is a **snapshot, not a live feed**: entries that already exist come from earlier runs. Rerun the tail cell during or after a LIVE launch to see new lines; Step 4 shows the same callbacks scoped to each trial.

<details><summary>Optional: stream callbacks live in a terminal</summary>

The inline tail is a snapshot. For a continuously updating feed during a LIVE launch, open a terminal in the repo and run the command for your row's sink:

- Web server rows: `tail -n 0 -F results/web_server.out`
- Lab exfil listener: `tail -n 0 -F results/listeners/<lab>.log`
</details>

**Next:** Step 3 launches the attack (LIVE) or loads it (REPLAY).

In [ ]:
display(HTML(op.setup_html(S, DEMO_ROW, RUN_LIVE, REPEAT)))
display(HTML(demo.listener_html()))
display(HTML(demo.watch_html()))

<a id="execute"></a>
### Step 3 · Launch the attack, or replay it

**Done so far:** the attack and its sink are understood, and the setup is checked.

**Do:** run the cell. **This is the only cell in Part A that launches anything.**

**Expected:**
- **REPLAY:** returns at once with the stored run ID and trial count.
- **LIVE:** lints and validates the config, then runs the row and its app-control twin `REPEAT` times each. This makes real LLM calls and takes several minutes. When it finishes, it prints the last harness log lines and the new run ID. Do not edit configs or `.env` while it runs.

**Next:** Step 4.

In [ ]:
demo_result = demo.execute(S, live=RUN_LIVE, repeat=REPEAT)

<a id="verdict-proof"></a>
### Step 4 · Read the verdict and its proof

**Done so far:** the attack was launched, or a stored run was loaded.

**Do:** run the cell. It only displays the result and launches nothing.

**Expected, top to bottom:**
1. **Verdict:** the outcome, trial counts and the three rates: effect rate (landed / valid), detection rate (on-target detections / delivered) and NOT-RUN rate. One landed trial makes the row vulnerable.
2. **Trace** of the featured trial (the first landing, if any): user → model input → enforcement → tool calls → reply → sink.
3. **Evidence per attempt** (expand it): full model calls, subprocess output, and OOB / sink log lines captured during that trial, marked "Oracle matched" where they hit.

A blocked attempt, a model refusal and an undelivered injection are different outcomes. Log lines are observations, not verdicts. Runs made before evidence capture show "unavailable" because they did not record it; that does not mean no callback occurred.

**Next:** the demo ends here. To compare the twins or open other trials, use Part B · B5 Results.

In [ ]:
display(HTML(demo.verdict_html(demo_result)))
display(HTML(demo.listener_html(demo_result)))

<a id="operator-console"></a>
## Part B · Operator console (optional)

**Done so far:** the guided demo is complete.

Use this part to browse rows (B2), change settings for the next launch (B3), run batches (B4) and inspect results (B5–B6). Displaying it launches nothing. Only B4 launches runs, either by its buttons or when `OPERATOR_AUTORUN` is set.

- **Scratch** runs stay in `results/scratch/`. Lab state and some listener files are shared.
- **Real** runs need confirmation and are permanent (append-only).

In [ ]:
display(HTML("<details><summary>Operator workflow</summary>" + op.flow_html() + "</details>"))
display(HTML("<details><summary>Available demo rows</summary>" + op.demo_rows_html() + "</details>"))
display(HTML("<details><summary>Outcome reference</summary>" + op.legend_html() + "</details>"))

### B1 · Environment
Tools, stores and whether a run is already going. `.env` is reported as present or missing, never shown.

In [ ]:
display(HTML(op.environment_html()))

### B2 · Test cases
Every TC-ID of the workbook (v4.2) with its coverage status, then one card per TC-ID: workbook objective
and success criterion, harness, each row's attack / oracle / delivery proof / twin / baseline, what a
vendor sees and must do, what the outcomes mean, how to run it and its latest stored result.

In [ ]:
display(HTML(op.overview_html()))

#### Inspect one test case
**Do:** select a TC-ID to read its card.
**Next:** to demo one of its rows, copy the row name into `DEMO_ROW` in Part A and rerun Part A. Selecting here changes nothing else.

In [ ]:
display(op.catalog_ui())

### B3 · Settings
Every variable the harness reads, with its value and source. Non-secret values can be overridden **for the next launch only**, because `.env` is never written. Secrets and header values show only as set / not set. `EXTRA_HEADERS` shows only endpoints and header names.

**Next:** after an override, rerun Part A from Step 2 to check the setup before a LIVE launch.

In [ ]:
display(op.settings_ui(S))

### B4 · Run
Pick a TC-ID, a row or a whole config; the FILTER is filled in and `run_evals.sh` widens it to every
condition of each row, so a twin and its baseline share a run_id.

- **Scratch** (default): runs land in `results/scratch/` and its own `scratch.db`. The real store,
  `results/runs/` and `results/<config>.*` are untouched.
- **Real**: appended to `results/guardrail_eval.db` for good. Tick the confirmation first.
- One run at a time (RAG rows share `chroma_db`, lab rows share the `:9999` listener).
- `run_evals.sh` lints the prompts before each config; **Lint + validate** shows it beforehand.
- **Stop** sends SIGKILL to this run's process group only (never `pkill -f`) and marks the run aborted.
- Do not edit configs, datasets or `.env` while a run is going.
- A run counts only if it wrote results with graded rows; the exit code is not trusted.

In [ ]:
display(op.run_ui(S))

#### Headless launch (verification)
Set `OPERATOR_AUTORUN=<config>:<FILTER>` to launch from here without clicking, e.g.
`OPERATOR_AUTORUN=tools:CTRL-delete jupyter nbconvert --execute …`. Scratch only. With it unset this cell does nothing.

In [ ]:
op.autorun(S)

### B5 · Results
Pick a run (this session's first, then scratch and real). Per row: state, outcome counts, the three rates
(effect = landed / valid, detection = on-target events / delivered, NOT-RUN rate), held_by / hop and the
INCONCLUSIVE / STALE flags. Twins are compared only with the baseline from the same run_id. Then
drill into any trial: delivery proof, tool calls, reply, sink.

In [ ]:
display(op.results_ui(S))

### B6 · Store
`scripts/db_report.py` on the real store: coverage with the latest result per TC-ID, then every run. Per-row detail: `db_report.py --rows`.

In [ ]:
print(op.store_report(op.REAL_DB))
print()
print(op.store_report(op.REAL_DB, "--runs"))

### B7 · Known issues
Open items, shown as they are; none is fixed by this notebook.

In [ ]:
display(HTML(op.known_issues_html()))